In [ ]:
from google.colab import files
uploaded = files.upload()

Saving twitter_algoritmo.pdf to twitter_algoritmo.pdf
Saving scaling_laws_llm.pdf to scaling_laws_llm.pdf
Saving retrieval_augmented_generation.pdf to retrieval_augmented_generation.pdf
Saving lora_low_rank_adaptation.pdf to lora_low_rank_adaptation.pdf
Saving llama_foundation_models.pdf to llama_foundation_models.pdf
Saving instruct_gpt.pdf to instruct_gpt.pdf
Saving gpt4_technical_report.pdf to gpt4_technical_report.pdf
Saving gpt3_language_models.pdf to gpt3_language_models.pdf
Saving escrita_academica_ia.pdf to escrita_academica_ia.pdf
Saving bioetica_e_ia.pdf to bioetica_e_ia.pdf
Saving bert_pretraining.pdf to bert_pretraining.pdf
Saving attention_is_all_you_need.pdf to attention_is_all_you_need.pdf


In [ ]:
import os
print(os.listdir('/content'))

['.config', 'llama_foundation_models.pdf', 'attention_is_all_you_need.pdf', 'gpt4_technical_report.pdf', 'twitter_algoritmo.pdf', 'retrieval_augmented_generation.pdf', 'gpt3_language_models.pdf', 'scaling_laws_llm.pdf', 'lora_low_rank_adaptation.pdf', 'escrita_academica_ia.pdf', 'bert_pretraining.pdf', 'instruct_gpt.pdf', 'bioetica_e_ia.pdf', 'sample_data']


In [ ]:
# 1. Ler os arquivos .pdf e separar linha por linha
arquivos_md = ['twitter_algoritmo.pdf', 'escrita_academica_ia.pdf', 'bioetica_e_ia.pdf']

trechos = []  # cada item: {"texto": ..., "arquivo": ...}

for nome_arquivo in arquivos_md:
    with open(nome_arquivo, 'r', encoding='utf-8') as f:
        linhas = f.readlines()
    for linha in linhas:
        linha_limpa = linha.strip()
        if linha_limpa:  # ignora linhas vazias
            trechos.append({"texto": linha_limpa, "arquivo": nome_arquivo})

print(f"Total de linhas não-vazias lidas: {len(trechos)}")
print(f"\nExemplo de trecho:")
print(trechos[0])

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb5 in position 11: invalid start byte

In [ ]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.7 MB/s eta 0:00:00


In [ ]:
from PyPDF2 import PdfReader

# 1. Ler os arquivos .pdf e separar linha por linha
arquivos_pdf = ['twitter_algoritmo.pdf', 'escrita_academica_ia.pdf', 'bioetica_e_ia.pdf']

trechos = []  # cada item: {"texto": ..., "arquivo": ...}

for nome_arquivo in arquivos_pdf:
    try:
        reader = PdfReader(nome_arquivo)
        for page in reader.pages:
            text = page.extract_text()
            if text:
                for linha in text.split('\n'):
                    linha_limpa = linha.strip()
                    if linha_limpa:  # ignora linhas vazias
                        trechos.append({"texto": linha_limpa, "arquivo": nome_arquivo})
    except Exception as e:
        print(f"Erro ao processar o arquivo {nome_arquivo}: {e}")

print(f"Total de linhas não-vazias lidas: {len(trechos)}")
if trechos:
    print(f"\nExemplo de trecho:")
    print(trechos[0])
else:
    print("Nenhum trecho extraído.")

Total de linhas não-vazias lidas: 2191

Exemplo de trecho:
{'texto': 'ARTIGOS                                                            E-ISSN: 2176 -6665', 'arquivo': 'twitter_algoritmo.pdf'}


In [ ]:
!pip install openai

In [1]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get('OPENAI_API_KEY'),
    base_url="https://openrouter.ai/api/v1"
   )

In [2]:
def gerar_embedding(texto):
    response = client.embeddings.create(
        model="nvidia/nemotron-3-embed-1b:free",
        input=texto,
        encoding_format="float"
    )
    return response.data[0].embedding, response.usage

In [6]:
# 1. Instalar bibliotecas necessárias
!pip install -q PyPDF2 langchain-text-splitters tiktoken pandas

import os
import glob
import tiktoken
import pandas as pd
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import files

# 2. Localizar arquivos PDF na pasta; se não houver nenhum, solicitar upload
arquivos_pdf = glob.glob("*.pdf")

if len(arquivos_pdf) == 0:
    print("Nenhum arquivo PDF encontrado na sessão. Por favor, envie os arquivos abaixo:")
    uploaded = files.upload()
    arquivos_pdf = glob.glob("*.pdf")

print(f"\nArquivos encontrados para análise: {arquivos_pdf}")

# 3. Extrair todo o texto dos PDFs encontrados
texto_completo = ""
for nome_arquivo in arquivos_pdf:
    try:
        reader = PdfReader(nome_arquivo)
        for page in reader.pages:
            conteudo = page.extract_text()
            if conteudo:
                texto_completo += conteudo + "\n"
    except Exception as e:
        print(f"Erro ao ler {nome_arquivo}: {e}")

print(f"Total de caracteres extraídos dos PDFs: {len(texto_completo):,}")

# 4. Configuração dos Testes de 1 a 6
encoder = tiktoken.get_encoding("cl100k_base")

testes = [
    {"teste": "Teste 1", "chunk_size": 200, "chunk_overlap": 0},
    {"teste": "Teste 2", "chunk_size": 500, "chunk_overlap": 0},
    {"teste": "Teste 3", "chunk_size": 1000, "chunk_overlap": 0},
    {"teste": "Teste 4", "chunk_size": 2000, "chunk_overlap": 0},
    {"teste": "Teste 5", "chunk_size": 500, "chunk_overlap": 50},
    {"teste": "Teste 6", "chunk_size": 500, "chunk_overlap": 200},
]

metricas = []

# 5. Execução do Chunking e Cálculo das Métricas
for t in testes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=t["chunk_size"],
        chunk_overlap=t["chunk_overlap"],
        length_function=len
    )

    chunks = splitter.split_text(texto_completo)
    total_chunks = len(chunks)

    if total_chunks > 0:
        tamanhos = [len(c) for c in chunks]
        tam_min = min(tamanhos)
        tam_max = max(tamanhos)
        tam_medio = sum(tamanhos) / total_chunks

        # Contagem de tokens
        tokens_lista = [len(encoder.encode(c)) for c in chunks]
        total_tokens = sum(tokens_lista)
        tokens_medio = total_tokens / total_chunks

        # Chunks sobrepostos
        chunks_sobrepostos = (total_chunks - 1) if t["chunk_overlap"] > 0 and total_chunks > 1 else 0
        perc_overlap = (t["chunk_overlap"] / t["chunk_size"]) * 100
    else:
        tam_min = tam_max = tam_medio = total_tokens = tokens_medio = chunks_sobrepostos = perc_overlap = 0

    metricas.append({
        "Teste": t["teste"],
        "Chunk Size": t["chunk_size"],
        "Overlap": t["chunk_overlap"],
        "Total Chunks": total_chunks,
        "Tam. Médio (chars)": round(tam_medio, 1),
        "Tam. Mín (chars)": tam_min,
        "Tam. Máx (chars)": tam_max,
        "Chunks Sobrepostos": chunks_sobrepostos,
        "% Overlap": f"{perc_overlap:.1f}%",
        "Total Tokens": total_tokens,
        "Tokens Médio/Chunk": round(tokens_medio, 1)
    })

# 6. Exibir a tabela com os resultados calculados
df_resultados = pd.DataFrame(metricas)
display(df_resultados)

# 7. Salvar e baixar a planilha com as métricas
df_resultados.to_csv("metricas_testes_1_a_6.csv", index=False)
files.download("metricas_testes_1_a_6.csv")


Arquivos encontrados para análise: ['attention_is_all_you_need.pdf', 'bert_pretraining.pdf', 'gpt3_language_models.pdf', 'bioetica_e_ia.pdf']
Total de caracteres extraídos dos PDFs: 393,155


,Teste,Chunk Size,Overlap,Total Chunks,Tam. Médio (chars),Tam. Mín (chars),Tam. Máx (chars),Chunks Sobrepostos,% Overlap,Total Tokens,Tokens Médio/Chunk
0,Teste 1,200,0,2675,145.9,30,199,0,0.0%,106755,39.9
1,Teste 2,500,0,855,458.7,382,499,0,0.0%,108001,126.3
2,Teste 3,1000,0,411,955.5,485,999,0,0.0%,108314,263.5
3,Teste 4,2000,0,201,1954.9,1452,1999,0,0.0%,108437,539.5
4,Teste 5,500,50,881,458.3,336,499,880,10.0%,111630,126.7
5,Teste 6,500,200,1283,457.8,305,499,1282,40.0%,162948,127.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>